In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP
# ==============================================================================
!pip install -q pykoopman scikit-learn numpy pandas torch sympy networkx datasets huggingface_hub kagglehub scipy

In [ ]:
# ==============================================================================
# CELL 2: DATA INGESTION & FRACTAL FEATURE ENGINEERING
# ==============================================================================
import pandas as pd
import numpy as np
from scipy.integrate import odeint
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("📡 GENERATING CHAOTIC SYSTEM & MULTI-SCALE TEMPORAL FEATURES")
print("="*80)

# 1. Generate Lorenz Attractor (Navier-Stokes Fluid Proxy)
def lorenz_system(state, t, sigma=10.0, beta=8./3, rho=28.0):
    x, y, z = state
    return [sigma * (y - x), x * (rho - z) - y, x * y - beta * z]

t_steps = np.linspace(0, 50, 5000)
fluid_states = odeint(lorenz_system, [1.0, 1.0, 1.0], t_steps)
df_physics = pd.DataFrame(fluid_states, columns=['x_pos', 'y_pos', 'z_pos'])

# Target: Complex emergent property (Requires ratios, derivatives, and synergy to solve)
df_physics['System_Energy'] = 0.5 * (df_physics['x_pos']**2) / (df_physics['z_pos'] + 1e-5)

# --- FRACTAL FEATURE ENGINEERING ENGINE ---
print(">> Engineering Time-Derivatives, Ratios, and Multi-Scale Volatility...")
base_cols = ['x_pos', 'y_pos', 'z_pos']

# A. Time Derivatives (Velocity, Acceleration)
for col in base_cols:
    df_physics[f'{col}_dt1'] = df_physics[col].diff()          # 1st Derivative (Velocity)
    df_physics[f'{col}_dt2'] = df_physics[col].diff().diff()   # 2nd Derivative (Acceleration)

# B. Multi-Scale Temporal Groupings (Small t, Large t)
time_windows = [3, 15, 50] # Short, Medium, Long-term groupings
for col in base_cols:
    for w in time_windows:
        # Standard Deviation from Average (Volatility / Energy proxy)
        df_physics[f'{col}_std_t{w}'] = df_physics[col].rolling(window=w).std()
        df_physics[f'{col}_mean_t{w}'] = df_physics[col].rolling(window=w).mean()

# C. Divisibility / Ratio Relationships (Cross-Combinations)
for i, col1 in enumerate(base_cols):
    for col2 in base_cols[i+1:]:
        df_physics[f'ratio_{col1}_{col2}'] = df_physics[col1] / (df_physics[col2] + 1e-5)

# D. Inject Noise to test robustness
for i in range(15): # Reduced to 15 to balance the massive new feature space
    df_physics[f'quantum_noise_{i}'] = np.random.normal(0, 5, 5000)

df_physics = df_physics.dropna().reset_index(drop=True)
print(f"✅ Multi-Scale Matrix Ready. Shape: {df_physics.shape} (Features engineered)")
print("="*80)

ACTIVE_DF = df_physics.copy()
TARGET_VAR = 'System_Energy'

In [ ]:
# ==============================================================================
# CELL 3: DYNAMIC APRIORI SYNERGY ENGINE (COMPUTE-AWARE)
# ==============================================================================
import torch
from sklearn.kernel_approximation import Nystroem
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

class InfiniteHilbertProjector:
    def __init__(self, components=100, gamma=0.1):
        self.scaler = StandardScaler()
        self.kernel_map = Nystroem(kernel='rbf', gamma=gamma, n_components=components)
        self.is_fitted = False
        
    def lift(self, X):
        X_scaled = self.scaler.fit_transform(X) if not self.is_fitted else self.scaler.transform(X)
        if not self.is_fitted:
            self.kernel_map.fit(X_scaled)
            self.is_fitted = True
        projected = torch.tensor(self.kernel_map.transform(X_scaled), dtype=torch.float32)
        return projected / (torch.norm(projected, dim=1, keepdim=True) + 1e-8)

class DynamicSynergyBeamSearch:
    def __init__(self, target_y, max_beam_width=15, max_order=10):
        y_np = target_y.values.reshape(-1, 1)
        self.y = torch.tensor(StandardScaler().fit_transform(y_np), dtype=torch.float32)
        self.max_beam_width = max_beam_width
        self.max_order = max_order
        
    def _approximate_synergy(self, X_tuple_tensor):
        cov_matrix = torch.matmul(X_tuple_tensor.T, self.y) / len(self.y)
        return torch.sum(cov_matrix ** 2).item() * 100

    def run_beam_search(self, feature_dict):
        print(f"🚀 Initiating Compute-Aware Synergy Beam Search")
        print(f"   Max Order: {self.max_order} | Starting Beam Width: {self.max_beam_width}")
        print("-" * 80)
        
        current_beam = [([f], self._approximate_synergy(feature_dict[f])) for f in feature_dict.keys()]
        current_beam = sorted(current_beam, key=lambda x: x[1], reverse=True)[:self.max_beam_width]
        best_overall = current_beam[0]
        
        for order in range(1, self.max_order + 1):
            top_score = current_beam[0][1]
            bottom_score = current_beam[-1][1]
            beam_dropoff = top_score - bottom_score
            
            # --- DYNAMIC COMPUTE ALLOCATION (PRUNING) ---
            # If the dropoff is too steep, the bottom candidates are dead weight. Cut them.
            active_beam_width = len(current_beam)
            if beam_dropoff > (top_score * 0.5) and active_beam_width > 3:
                # Keep only candidates within 50% of the top score
                current_beam = [c for c in current_beam if c[1] >= (top_score * 0.5)]
                print(f"   ⚡ COMPUTE SAVED: Beam width dynamically pruned from {active_beam_width} down to {len(current_beam)}")
            
            print(f"🟢 ORDER {order} ANALYSIS (Active Beam: {len(current_beam)}):")
            print(f"   ↳ Top Node:   {current_beam[0][0]} (Score: {top_score:.4f})")
            
            next_candidates = []
            for state, score in current_beam:
                for f in feature_dict.keys():
                    if f not in state:
                        candidate = sorted(list(state) + [f])
                        if candidate not in [x[0] for x in next_candidates]:
                            joint_tensor = torch.prod(torch.stack([feature_dict[var] for var in candidate]), dim=0)
                            joint_tensor = joint_tensor / (torch.norm(joint_tensor, dim=1, keepdim=True) + 1e-8)
                            syn_score = self._approximate_synergy(joint_tensor)
                            next_candidates.append((candidate, syn_score))
            
            if not next_candidates:
                break
                
            current_beam = sorted(next_candidates, key=lambda x: x[1], reverse=True)[:self.max_beam_width]
            
            # Convergence Check
            if current_beam[0][1] <= best_overall[1] * 1.01:
                print(f"\n🛑 Convergence Reached: Order {order+1} did not yield strictly higher emergent synergy.")
                break
                
            best_overall = current_beam[0] if current_beam[0][1] > best_overall[1] else best_overall
            print("-" * 80)
            
        return best_overall

# ==============================================================================
# EXECUTION: CAUSAL DISCOVERY
# ==============================================================================
print("\n" + "="*80)
print("🌌 TITAN PHYSICS ENGINE: RUNNING MULTI-SCALE CAUSAL MAPPING")
print("="*80)

X_cols = [c for c in ACTIVE_DF.columns if c != TARGET_VAR]
Y = ACTIVE_DF[TARGET_VAR]

print(">> Lifting Observables to Infinite Reproducing Kernel Hilbert Space...")
feature_tensors = {}
for col in X_cols:
    projector = InfiniteHilbertProjector(components=150)
    feature_tensors[col] = projector.lift(ACTIVE_DF[[col]].values)

# Widen beam to 15, let the dynamic pruner handle efficiency
beam_search = DynamicSynergyBeamSearch(target_y=Y, max_beam_width=15, max_order=6)
best_cluster, score = beam_search.run_beam_search(feature_tensors)

print("\n" + "="*80)
print("🔬 FINAL MULTI-SCALE DISCOVERY REPORT")
print("="*80)
print(f"✅ Discovered Governing Manifold:")
for feature in best_cluster:
    print(f"   • {feature}")
print(f"\n✅ Maximum Synergistic Interaction Score: {score:.4f}")
print("="*80)

In [ ]:
# ==============================================================================
# CELL 4: SYMBOLIC DISTILLATION (UNRESTRICTED EVOLUTION)
# ==============================================================================
import sys
try:
    from gplearn.genetic import SymbolicRegressor
    import sympy as sp
except ImportError:
    import subprocess
    print(">> Installing gplearn and sympy...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gplearn", "sympy"])
    from gplearn.genetic import SymbolicRegressor
    import sympy as sp

print("\n" + "="*80)
print("📐 STAGE 3: SYMBOLIC DISTILLATION ENGINE (UNRESTRICTED)")
print("="*80)

# Safety Fallback
if 'best_cluster' not in locals() or 'best_cluster' not in globals():
    best_cluster = ['y_pos_std_t3', 'z_pos_dt1', 'z_pos_mean_t50', 'z_pos_std_t3']
    
if 'ACTIVE_DF' not in locals() or 'ACTIVE_DF' not in globals():
    raise NameError("❌ ACTIVE_DF is missing! Please re-run Cell 2.")

print(f">> Extracting Target and Discovered Manifold: {best_cluster}")

X_manifold = ACTIVE_DF[best_cluster].values
y_target = ACTIVE_DF[TARGET_VAR].values

# --- THE FIX: UNRESTRICTED GENETIC PARAMETERS ---
est_gp = SymbolicRegressor(
    population_size=3500,        # More equations per generation
    generations=25,              # Evolve for longer (25 generations instead of 15)
    stopping_criteria=0.001,     # Demand higher accuracy before stopping
    p_crossover=0.7, 
    p_subtree_mutation=0.1,
    p_hoist_mutation=0.05, 
    p_point_mutation=0.1,
    max_samples=0.9, 
    verbose=1,
    parsimony_coefficient=0.0005, # Loosened Occam's Razor to allow fractions!
    random_state=42,
    function_set=('add', 'sub', 'mul', 'div', 'abs') # Added 'abs' to protect denominators
)

print("\n>> Evolving Mathematical Equations (Genetic Programming)...")
est_gp.fit(X_manifold, y_target)

raw_equation = str(est_gp._program)
for i, feature_name in enumerate(best_cluster):
    raw_equation = raw_equation.replace(f'X{i}', feature_name)

print("\n" + "="*80)
print("🏆 FINAL DISCOVERED LAW OF PHYSICS")
print("="*80)
print(f"System_Energy ≈ \n\n{raw_equation}\n")
print("="*80)

In [ ]:
# ==============================================================================
# CELL 5: MATHEMATICAL VERIFICATION & ALTERNATE EQUATIONS
# ==============================================================================
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

print("="*80)
print("🔬 FINAL VERIFICATION & VALIDATION PROTOCOL")
print("="*80)

# 1. Test the Best Equation's Validity
y_pred = est_gp.predict(X_manifold)
r_squared = r2_score(y_target, y_pred)
mae = mean_absolute_error(y_target, y_pred)

print(f"✅ Best Equation Validity Score (R²): {r_squared:.4f}")
print(f"✅ Mean Absolute Error: {mae:.4f}")

# 2. Extract Alternate Equations (The Evolutionary Hall of Fame)
print("\n>> Extracting Alternate Evolutionary Equations...")
unique_programs = {}
# Scan the final generation for the best diverse equations
for program in est_gp._programs[-1]:
    if program is not None:
        eq_str = str(program)
        for i, feature in enumerate(best_cluster):
            eq_str = eq_str.replace(f'X{i}', feature)
        # Store by fitness to find the best unique ones
        unique_programs[eq_str] = program.fitness_

# Sort by fitness (lowest error is best)
sorted_programs = sorted(unique_programs.items(), key=lambda x: x[1])

print("\n🏆 TOP 5 DISCOVERED EQUATIONS (Complexity vs Accuracy):")
for i, (eq, error) in enumerate(sorted_programs[:5]):
    print(f"  {i+1}. Error: {error:.4f} | Eq: {eq}")

# 3. Visual Proof (True Physics vs Discovered Physics)
plt.figure(figsize=(10, 6))
plt.scatter(y_target, y_pred, alpha=0.5, c='cyan', edgecolors='blue', label='Discovered Law')
plt.plot([min(y_target), max(y_target)], [min(y_target), max(y_target)], 'r--', lw=2, label='Perfect Physics (Truth)')
plt.title(f"Visual Proof: Discovered Equation vs True System Energy\nR² = {r_squared:.4f}", fontsize=14)
plt.xlabel("True Underlying System Energy", fontsize=12)
plt.ylabel("AI Predicted System Energy", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("="*80)